In [ ]:
# ============================================================
# Experiment A
# Stage-Matched Patient-Level Classification
#
# Positive:
#   desaturated patient의 desaturation EEG epoch
#   -> label 1
#
# Negative:
#   undesaturated patient의 normal sleep EEG epoch
#   -> label 0
#
# 핵심:
#   Negative epoch의 Sleep Stage 분포를
#   Positive epoch의 Sleep Stage 분포에 맞춤
#
# Sleep stages:
#   N1 / N2 / N3 / REM
#
# Patient-level:
#   4-Fold Cross Validation
#
# Positive patients:
#   64명
#
# Negative patients:
#   7명
#
# Fold:
#   Test  = 현재 fold
#   Val   = 다음 fold
#   Train = 나머지 2개 fold
#
# Train:
#   Positive patient당 최대 30 epoch
#   Negative patient당 최대 30 epoch 후보 생성 후
#   Positive stage distribution에 맞춰 sampling
#
# Validation:
#   Positive 전체 epoch
#   Negative는 Positive stage distribution에 맞춰 sampling
#
# Test:
#   Positive 전체 epoch
#   Negative는 Positive stage distribution에 맞춰 sampling
#
# Preprocessing:
#   60 Hz / 120 Hz Butterworth band-stop
#   STFT (Hann)
#
# Model:
#   ResNet18
#
# Best model:
#   Validation Balanced Accuracy
#
# Final:
#   Test Balanced Accuracy
#   Test AUC
# ============================================================


# ============================================================
# 0. Import
# ============================================================

import os
import gc
import copy
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

from torch.utils.data import Dataset, DataLoader

from scipy.signal import butter, filtfilt, stft

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. Random Seed
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("Experiment A")
print("Stage-Matched Patient-Level Classification")
print("=" * 70)

print()
print("Device:", DEVICE)


# ============================================================
# 3. Training Settings
# ============================================================

BATCH_SIZE = 16

EPOCHS = 100

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4

MAX_EPOCHS_PER_PATIENT = 30

N_FOLDS = 4


# ============================================================
# 4. EEG / STFT Settings
# ============================================================

SFREQ = 256

NPERSEG = 256

NOVERLAP = 128


EEG_CHANNELS = [
    "EEG F3-M2",
    "EEG F4-M1",
    "EEG C3-M2",
    "EEG C4-M1",
    "EEG O1-M2",
    "EEG O2-M1"
]


TARGET_STAGES = [
    "N1",
    "N2",
    "N3",
    "REM"
]


# ============================================================
# 5. Paths
# ============================================================

BASE_FOLDER = (
    "/content/drive/MyDrive/"
    "Sleep_Project/example"
)


# ------------------------------------------------------------
# Positive
# ------------------------------------------------------------

POSITIVE_FOLDERS = [

    os.path.join(
        BASE_FOLDER,
        "positive_patient_data"
    ),

    os.path.join(
        BASE_FOLDER,
        "positive_patient_data_3"
    )
]


# ------------------------------------------------------------
# Negative
# ------------------------------------------------------------

NEGATIVE_FOLDERS = [

    os.path.join(
        BASE_FOLDER,
        "negative_patient_data"
    ),

    os.path.join(
        BASE_FOLDER,
        "negative_patient_data_3"
    ),

    os.path.join(
        BASE_FOLDER,
        "negative_patient_data_5"
    )
]


# ============================================================
# 6. Stage normalization
# ============================================================

def normalize_stage(stage):

    if pd.isna(stage):
        return None

    s = str(stage).strip()

    mapping = {

        "N1": "N1",
        "N2": "N2",
        "N3": "N3",

        "REM": "REM",
        "R": "REM",

        "Sleep stage 1": "N1",
        "Sleep stage 2": "N2",
        "Sleep stage 3": "N3",
        "Sleep stage R": "REM",

        "Sleep stage N1": "N1",
        "Sleep stage N2": "N2",
        "Sleep stage N3": "N3"
    }

    return mapping.get(
        s,
        None
    )


# ============================================================
# 7. Collect NPY files
# ============================================================

def collect_npy_files(folders):

    result = {}

    for folder in folders:

        print()
        print(
            "Checking folder:"
        )

        print(
            folder
        )

        if not os.path.exists(folder):

            print(
                "  WARNING: Folder not found"
            )

            continue

        npy_files = [
            f
            for f in os.listdir(folder)
            if f.endswith(".npy")
            and "metadata" not in f.lower()
        ]

        print(
            f"  NPY files: {len(npy_files)}"
        )

        for filename in sorted(npy_files):

            patient_id = filename[:-4]

            full_path = os.path.join(
                folder,
                filename
            )

            # 같은 patient가 여러 폴더에 있으면
            # 첫 번째 파일만 사용
            if patient_id not in result:

                result[
                    patient_id
                ] = full_path

    return result


# ============================================================
# 8. Collect Positive / Negative
# ============================================================

positive_files = collect_npy_files(
    POSITIVE_FOLDERS
)

negative_files = collect_npy_files(
    NEGATIVE_FOLDERS
)


positive_patients = sorted(
    positive_files.keys()
)

negative_patients = sorted(
    negative_files.keys()
)


# ============================================================
# 9. Patient 확인
# ============================================================

print()
print("=" * 70)
print("[1] Patient 확인")
print("=" * 70)

print()
print(
    "Positive patients:",
    len(positive_patients)
)

print(
    "Negative patients:",
    len(negative_patients)
)


print()
print(
    "Negative patient list:"
)

for p in negative_patients:

    print(
        " ",
        p
    )


# ============================================================
# 10. Patient validation
# ============================================================

if len(positive_patients) < N_FOLDS:

    raise ValueError(
        "Positive patient 수가 4보다 적습니다."
    )


if len(negative_patients) < N_FOLDS:

    raise ValueError(
        "Negative patient 수가 4보다 적습니다."
    )


# ------------------------------------------------------------
# Positive / Negative overlap
# ------------------------------------------------------------

overlap = (
    set(positive_patients)
    &
    set(negative_patients)
)


if len(overlap) > 0:

    print()
    print(
        "ERROR: Positive / Negative overlap"
    )

    print(
        sorted(overlap)
    )

    raise ValueError(
        "같은 patient가 Positive와 Negative에 동시에 존재합니다."
    )


# ============================================================
# 11. Load EEG + Stage
# ============================================================

def load_all_patients(
    patient_files,
    label
):

    X_list = []

    y_list = []

    patient_list = []

    stage_list = []

    epoch_number_list = []


    for patient_id in sorted(
        patient_files.keys()
    ):

        npy_path = patient_files[
            patient_id
        ]


        print()
        print(
            f"Loading {patient_id}"
        )


        # ----------------------------------------------------
        # NPY
        # ----------------------------------------------------

        eeg = np.load(
            npy_path
        )


        print(
            "  EEG shape:",
            eeg.shape
        )


        # ----------------------------------------------------
        # Shape check
        # ----------------------------------------------------

        if eeg.ndim != 3:

            print(
                "  SKIP: ndim != 3"
            )

            continue


        if eeg.shape[1] != 6:

            print(
                "  SKIP: channel != 6"
            )

            continue


        n_epochs = eeg.shape[0]


        # ----------------------------------------------------
        # CSV metadata
        # ----------------------------------------------------

        folder = os.path.dirname(
            npy_path
        )


        csv_path = os.path.join(
            folder,
            patient_id + ".csv"
        )


        if not os.path.exists(csv_path):

            print(
                "  SKIP: CSV not found"
            )

            continue


        meta = pd.read_csv(
            csv_path
        )


        # ----------------------------------------------------
        # Stage column
        # ----------------------------------------------------

        if "stage" not in meta.columns:

            print(
                "  SKIP: stage column not found"
            )

            print(
                "  Columns:",
                list(meta.columns)
            )

            continue


        stages = (
            meta["stage"]
            .apply(normalize_stage)
            .values
        )


        # ----------------------------------------------------
        # Epoch count check
        # ----------------------------------------------------

        if len(stages) != n_epochs:

            print(
                "  SKIP: epoch count mismatch"
            )

            print(
                "  EEG epochs:",
                n_epochs
            )

            print(
                "  CSV rows:",
                len(stages)
            )

            continue


        # ----------------------------------------------------
        # Remove invalid stage
        # ----------------------------------------------------

        valid_mask = np.array(
            [
                s in TARGET_STAGES
                for s in stages
            ]
        )


        if not np.all(valid_mask):

            eeg = eeg[
                valid_mask
            ]

            stages = stages[
                valid_mask
            ]


        n_valid = len(
            stages
        )


        if n_valid == 0:

            print(
                "  SKIP: no valid stages"
            )

            continue


        # ----------------------------------------------------
        # Append
        # ----------------------------------------------------

        X_list.append(
            eeg.astype(
                np.float32
            )
        )


        y_list.append(
            np.full(
                n_valid,
                label,
                dtype=np.int64
            )
        )


        patient_list.extend(
            [patient_id] * n_valid
        )


        stage_list.extend(
            stages
        )


        epoch_number_list.extend(
            range(n_valid)
        )


        print(
            "  Valid epochs:",
            n_valid
        )


        print(
            "  Stage distribution:"
        )

        print(
            pd.Series(
                stages
            ).value_counts()
            .reindex(
                TARGET_STAGES,
                fill_value=0
            )
        )


    # ========================================================
    # Empty
    # ========================================================

    if len(X_list) == 0:

        return (
            np.empty(
                (
                    0,
                    6,
                    7680
                ),
                dtype=np.float32
            ),

            np.empty(
                0,
                dtype=np.int64
            ),

            np.empty(
                0,
                dtype=object
            ),

            np.empty(
                0,
                dtype=object
            ),

            np.empty(
                0,
                dtype=np.int64
            )
        )


    # ========================================================
    # Concatenate
    # ========================================================

    X = np.concatenate(
        X_list,
        axis=0
    )


    y = np.concatenate(
        y_list,
        axis=0
    )


    patients = np.array(
        patient_list,
        dtype=object
    )


    stages = np.array(
        stage_list,
        dtype=object
    )


    epoch_numbers = np.array(
        epoch_number_list,
        dtype=np.int64
    )


    return (
        X,
        y,
        patients,
        stages,
        epoch_numbers
    )


# ============================================================
# 12. Load Positive
# ============================================================

print()
print("=" * 70)
print("[2] Positive EEG loading")
print("=" * 70)


(
    X_pos_raw,
    y_pos_raw,
    patient_pos,
    stage_pos,
    epoch_pos
) = load_all_patients(
    positive_files,
    label=1
)


# ============================================================
# 13. Load Negative
# ============================================================

print()
print("=" * 70)
print("[3] Negative EEG loading")
print("=" * 70)


(
    X_neg_raw,
    y_neg_raw,
    patient_neg,
    stage_neg,
    epoch_neg
) = load_all_patients(
    negative_files,
    label=0
)


# ============================================================
# 14. Loaded data check
# ============================================================

print()
print("=" * 70)
print("[4] Loaded dataset")
print("=" * 70)


print()
print(
    "Positive EEG:",
    X_pos_raw.shape
)

print(
    "Negative EEG:",
    X_neg_raw.shape
)


print()
print(
    "Positive patients:",
    len(
        np.unique(
            patient_pos
        )
    )
)

print(
    "Negative patients:",
    len(
        np.unique(
            patient_neg
        )
    )
)


print()
print(
    "Positive stage distribution:"
)

print(
    pd.Series(
        stage_pos
    ).value_counts()
    .reindex(
        TARGET_STAGES,
        fill_value=0
    )
)


print()
print(
    "Negative stage distribution:"
)

print(
    pd.Series(
        stage_neg
    ).value_counts()
    .reindex(
        TARGET_STAGES,
        fill_value=0
    )
)


# ============================================================
# 15. Labels
# ============================================================

y_pos = np.ones(
    len(X_pos_raw),
    dtype=np.int64
)


y_neg = np.zeros(
    len(X_neg_raw),
    dtype=np.int64
)


# ============================================================
# 16. Combine raw data
# ============================================================

X_raw = np.concatenate(
    [
        X_pos_raw,
        X_neg_raw
    ],
    axis=0
)


y = np.concatenate(
    [
        y_pos,
        y_neg
    ],
    axis=0
)


patient_ids = np.concatenate(
    [
        patient_pos,
        patient_neg
    ],
    axis=0
)


stages = np.concatenate(
    [
        stage_pos,
        stage_neg
    ],
    axis=0
)


# ============================================================
# 17. Butterworth filters
# ============================================================

b60, a60 = butter(
    N=3,
    Wn=[59, 61],
    btype="bandstop",
    fs=SFREQ
)


b120, a120 = butter(
    N=3,
    Wn=[119, 121],
    btype="bandstop",
    fs=SFREQ
)


# ============================================================
# 18. STFT preprocessing
# ============================================================

def preprocess_and_stft(X):

    if len(X) == 0:

        return np.empty(
            (
                0,
                6,
                129,
                61
            ),
            dtype=np.float32
        )


    output = np.empty(
        (
            len(X),
            6,
            129,
            61
        ),
        dtype=np.float32
    )


    for i in range(
        len(X)
    ):

        if i % 100 == 0:

            print(
                f"STFT: {i}/{len(X)}"
            )


        for ch in range(6):

            signal = X[
                i,
                ch
            ]


            # 60 Hz
            signal = filtfilt(
                b60,
                a60,
                signal
            )


            # 120 Hz
            signal = filtfilt(
                b120,
                a120,
                signal
            )


            # STFT
            _, _, Zxx = stft(
                signal,
                fs=SFREQ,
                window="hann",
                nperseg=NPERSEG,
                noverlap=NOVERLAP
            )


            output[
                i,
                ch
            ] = np.abs(
                Zxx
            ).astype(
                np.float32
            )


    return output


# ============================================================
# 19. STFT
# ============================================================

print()
print("=" * 70)
print("[5] STFT")
print("=" * 70)


X_stft = preprocess_and_stft(
    X_raw
)


print()
print(
    "X_stft:",
    X_stft.shape
)


# ============================================================
# 20. Patient-level 4-fold
# ============================================================

rng = np.random.default_rng(
    SEED
)


pos_patient_list = np.array(
    positive_patients,
    dtype=object
)


neg_patient_list = np.array(
    negative_patients,
    dtype=object
)


rng.shuffle(
    pos_patient_list
)

rng.shuffle(
    neg_patient_list
)


POS_FOLDS = np.array_split(
    pos_patient_list,
    N_FOLDS
)


NEG_FOLDS = np.array_split(
    neg_patient_list,
    N_FOLDS
)


print()
print("=" * 70)
print("[6] Patient-level 4-fold")
print("=" * 70)


for i in range(
    N_FOLDS
):

    print()
    print(
        f"Fold {i + 1}"
    )

    print(
        "  Positive:",
        len(POS_FOLDS[i])
    )

    print(
        "  Negative:",
        len(NEG_FOLDS[i])
    )

    print(
        "  Negative patients:",
        list(
            NEG_FOLDS[i]
        )
    )


# ============================================================
# 21. Verify patient-level split
# ============================================================

all_pos_test = np.concatenate(
    POS_FOLDS
)


all_neg_test = np.concatenate(
    NEG_FOLDS
)


assert len(
    np.unique(
        all_pos_test
    )
) == len(
    positive_patients
)


assert len(
    np.unique(
        all_neg_test
    )
) == len(
    negative_patients
)


print()
print(
    "Patient split verification: OK"
)


# ============================================================
# 22. Helper:
#     patient별 최대 epoch sampling
# ============================================================

def sample_patient_epochs(
    indices,
    patient_array,
    max_per_patient,
    rng
):

    selected = []


    patients = np.unique(
        patient_array[
            indices
        ]
    )


    for patient in patients:

        patient_indices = indices[
            patient_array[
                indices
            ] == patient
        ]


        if len(
            patient_indices
        ) > max_per_patient:

            chosen = rng.choice(
                patient_indices,
                size=max_per_patient,
                replace=False
            )

        else:

            chosen = patient_indices


        selected.extend(
            chosen.tolist()
        )


    return np.array(
        selected,
        dtype=np.int64
    )


# ============================================================
# 23. Helper:
#     Stage distribution 확인
# ============================================================

def get_stage_counts(
    indices,
    stage_array
):

    counts = (
        pd.Series(
            stage_array[
                indices
            ]
        )
        .value_counts()
        .reindex(
            TARGET_STAGES,
            fill_value=0
        )
    )

    return counts.astype(
        int
    )


# ============================================================
# 24. Helper:
#     Stage-matched sampling
#
#     Positive stage distribution을 기준으로
#     Negative epoch을 sampling
# ============================================================

def stage_matched_sample(
    candidate_indices,
    candidate_stages,
    target_indices,
    target_stages,
    rng
):

    # --------------------------------------------------------
    # Positive target distribution
    # --------------------------------------------------------

    target_counts = (
        get_stage_counts(
            target_indices,
            target_stages
        )
    )


    target_total = (
        target_counts.sum()
    )


    if target_total == 0:

        raise ValueError(
            "Target Positive epoch이 없습니다."
        )


    # --------------------------------------------------------
    # Negative available distribution
    # --------------------------------------------------------

    candidate_counts = (
        get_stage_counts(
            candidate_indices,
            candidate_stages
        )
    )


    candidate_total = (
        candidate_counts.sum()
    )


    if candidate_total == 0:

        raise ValueError(
            "Negative candidate epoch이 없습니다."
        )


    # --------------------------------------------------------
    # Positive proportion
    # --------------------------------------------------------

    target_prop = (
        target_counts
        /
        target_total
    )


    # --------------------------------------------------------
    # 최대 feasible Negative total
    #
    # 각 stage의 Negative가 충분한지 확인
    # --------------------------------------------------------

    feasible_limits = []


    for stage in TARGET_STAGES:

        prop = target_prop[
            stage
        ]

        available = candidate_counts[
            stage
        ]


        if prop > 0:

            feasible_limits.append(
                available / prop
            )


    max_stage_matched_total = min(
        feasible_limits
    )


    # --------------------------------------------------------
    # Positive와 Negative 중
    # 가능한 만큼 사용
    # --------------------------------------------------------

    n_target = int(
        np.floor(
            min(
                target_total,
                candidate_total,
                max_stage_matched_total
            )
        )
    )


    if n_target <= 0:

        raise ValueError(
            "Stage matching 가능한 epoch이 없습니다."
        )


    # --------------------------------------------------------
    # 각 stage별 목표 개수
    # --------------------------------------------------------

    raw_counts = (
        target_prop
        *
        n_target
    )


    selected_counts = (
        np.floor(
            raw_counts
        )
        .astype(int)
    )


    # --------------------------------------------------------
    # 남은 epoch 배분
    # largest remainder
    # --------------------------------------------------------

    remainder = (
        n_target
        -
        selected_counts.sum()
    )


    fractional = (
        raw_counts
        -
        selected_counts
    )


    while remainder > 0:

        candidates = [

            stage

            for stage in TARGET_STAGES

            if (
                selected_counts[
                    stage
                ]
                <
                candidate_counts[
                    stage
                ]
            )
        ]


        if len(candidates) == 0:

            break


        stage = max(
            candidates,
            key=lambda s:
                fractional[s]
        )


        selected_counts[
            stage
        ] += 1


        fractional[
            stage
        ] = -1


        remainder -= 1


    # --------------------------------------------------------
    # 실제 sampling
    # --------------------------------------------------------

    selected = []


    for stage in TARGET_STAGES:

        n_select = int(
            selected_counts[
                stage
            ]
        )


        if n_select == 0:

            continue


        stage_candidates = (
            candidate_indices[
                candidate_stages[
                    candidate_indices
                ]
                ==
                stage
            ]
        )


        chosen = rng.choice(
            stage_candidates,
            size=n_select,
            replace=False
        )


        selected.extend(
            chosen.tolist()
        )


    selected = np.array(
        selected,
        dtype=np.int64
    )


    # --------------------------------------------------------
    # Shuffle
    # --------------------------------------------------------

    rng.shuffle(
        selected
    )


    return selected


# ============================================================
# 25. Dataset
# ============================================================

class EEGDataset(
    Dataset
):

    def __init__(
        self,
        X,
        y
    ):

        self.X = torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32
            )
        )


        self.y = torch.from_numpy(
            np.asarray(
                y,
                dtype=np.int64
            )
        )


    def __len__(
        self
    ):

        return len(
            self.y
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx]
        )


# ============================================================
# 26. ResNet18
# ============================================================

class StandardEEGResNet18(
    nn.Module
):

    def __init__(
        self,
        in_channels=6,
        num_classes=2
    ):

        super().__init__()


        self.resnet = (
            models.resnet18(
                weights=None
            )
        )


        self.resnet.conv1 = (
            nn.Conv2d(
                in_channels,
                64,
                kernel_size=7,
                stride=2,
                padding=3,
                bias=False
            )
        )


        self.resnet.fc = (
            nn.Linear(
                self.resnet.fc.in_features,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        return self.resnet(
            x
        )


# ============================================================
# 27. Fold training
# ============================================================

fold_results = []


for fold in range(
    N_FOLDS
):

    print()
    print("#" * 70)

    print(
        f"FOLD {fold + 1}/{N_FOLDS}"
    )

    print("#" * 70)


    # ========================================================
    # Patient split
    # ========================================================

    test_pos_patients = POS_FOLDS[
        fold
    ]


    test_neg_patients = NEG_FOLDS[
        fold
    ]


    val_fold = (
        fold + 1
    ) % N_FOLDS


    val_pos_patients = POS_FOLDS[
        val_fold
    ]


    val_neg_patients = NEG_FOLDS[
        val_fold
    ]


    train_pos_patients = np.concatenate(
        [
            POS_FOLDS[i]
            for i in range(N_FOLDS)
            if (
                i != fold
                and
                i != val_fold
            )
        ]
    )


    train_neg_patients = np.concatenate(
        [
            NEG_FOLDS[i]
            for i in range(N_FOLDS)
            if (
                i != fold
                and
                i != val_fold
            )
        ]
    )


    print()
    print(
        "Train Positive:",
        len(train_pos_patients)
    )

    print(
        "Val Positive:",
        len(val_pos_patients)
    )

    print(
        "Test Positive:",
        len(test_pos_patients)
    )


    print()
    print(
        "Train Negative:",
        len(train_neg_patients)
    )

    print(
        "Val Negative:",
        len(val_neg_patients)
    )

    print(
        "Test Negative:",
        len(test_neg_patients)
    )


    print()
    print(
        "Test Negative patients:"
    )

    print(
        list(
            test_neg_patients
        )
    )


    # ========================================================
    # Patient masks
    # ========================================================

    train_pos_indices = np.where(
        np.isin(
            patient_pos,
            train_pos_patients
        )
    )[0]


    train_neg_indices = np.where(
        np.isin(
            patient_neg,
            train_neg_patients
        )
    )[0]


    val_pos_indices = np.where(
        np.isin(
            patient_pos,
            val_pos_patients
        )
    )[0]


    val_neg_indices = np.where(
        np.isin(
            patient_neg,
            val_neg_patients
        )
    )[0]


    test_pos_indices = np.where(
        np.isin(
            patient_pos,
            test_pos_patients
        )
    )[0]


    test_neg_indices = np.where(
        np.isin(
            patient_neg,
            test_neg_patients
        )
    )[0]


    # ========================================================
    # TRAIN
    # ========================================================

    rng_train = np.random.default_rng(
        SEED + fold
    )


    # --------------------------------------------------------
    # Positive:
    # patient당 최대 30 epoch
    # --------------------------------------------------------

    sampled_train_pos = (
        sample_patient_epochs(
            train_pos_indices,
            patient_pos,
            MAX_EPOCHS_PER_PATIENT,
            rng_train
        )
    )


    # --------------------------------------------------------
    # Negative:
    # 먼저 patient당 최대 30 epoch
    # --------------------------------------------------------

    sampled_train_neg_candidates = (
        sample_patient_epochs(
            train_neg_indices,
            patient_neg,
            MAX_EPOCHS_PER_PATIENT,
            rng_train
        )
    )


    # --------------------------------------------------------
    # Negative stage matching
    # Positive TRAIN distribution 기준
    # --------------------------------------------------------

    sampled_train_neg = (
        stage_matched_sample(
            sampled_train_neg_candidates,
            stage_neg,
            sampled_train_pos,
            stage_pos,
            rng_train
        )
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    rng_val = np.random.default_rng(
        SEED + 1000 + fold
    )


    # Positive validation은 전체 사용
    # Negative는 Positive validation stage에 맞춤

    sampled_val_neg = (
        stage_matched_sample(
            val_neg_indices,
            stage_neg,
            val_pos_indices,
            stage_pos,
            rng_val
        )
    )


    # ========================================================
    # TEST
    # ========================================================

    rng_test = np.random.default_rng(
        SEED + 2000 + fold
    )


    # Positive test는 전체 사용
    # Negative는 Positive test stage에 맞춤

    sampled_test_neg = (
        stage_matched_sample(
            test_neg_indices,
            stage_neg,
            test_pos_indices,
            stage_pos,
            rng_test
        )
    )


    # ========================================================
    # Final indices
    # ========================================================

    train_indices = np.concatenate(
        [
            sampled_train_pos,
            sampled_train_neg
        ]
    )


    val_indices = np.concatenate(
        [
            val_pos_indices,
            sampled_val_neg
        ]
    )


    test_indices = np.concatenate(
        [
            test_pos_indices,
            sampled_test_neg
        ]
    )


    # ========================================================
    # Stage distribution 출력
    # ========================================================

    print()
    print("=" * 70)
    print("STAGE MATCHING CHECK")
    print("=" * 70)


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    train_pos_stage = get_stage_counts(
        sampled_train_pos,
        stage_pos
    )


    train_neg_stage = get_stage_counts(
        sampled_train_neg,
        stage_neg
    )


    print()
    print("TRAIN")

    print(
        "Positive:"
    )

    print(
        train_pos_stage
    )


    print(
        "Positive ratio:"
    )

    print(
        (
            train_pos_stage
            /
            train_pos_stage.sum()
        ).round(3)
    )


    print()
    print(
        "Negative:"
    )

    print(
        train_neg_stage
    )


    print(
        "Negative ratio:"
    )

    print(
        (
            train_neg_stage
            /
            train_neg_stage.sum()
        ).round(3)
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_pos_stage = get_stage_counts(
        val_pos_indices,
        stage_pos
    )


    val_neg_stage = get_stage_counts(
        sampled_val_neg,
        stage_neg
    )


    print()
    print("VALIDATION")

    print(
        "Positive:"
    )

    print(
        val_pos_stage
    )


    print(
        "Positive ratio:"
    )

    print(
        (
            val_pos_stage
            /
            val_pos_stage.sum()
        ).round(3)
    )


    print()
    print(
        "Negative:"
    )

    print(
        val_neg_stage
    )


    print(
        "Negative ratio:"
    )

    print(
        (
            val_neg_stage
            /
            val_neg_stage.sum()
        ).round(3)
    )


    # --------------------------------------------------------
    # Test
    # --------------------------------------------------------

    test_pos_stage = get_stage_counts(
        test_pos_indices,
        stage_pos
    )


    test_neg_stage = get_stage_counts(
        sampled_test_neg,
        stage_neg
    )


    print()
    print("TEST")

    print(
        "Positive:"
    )

    print(
        test_pos_stage
    )


    print(
        "Positive ratio:"
    )

    print(
        (
            test_pos_stage
            /
            test_pos_stage.sum()
        ).round(3)
    )


    print()
    print(
        "Negative:"
    )

    print(
        test_neg_stage
    )


    print(
        "Negative ratio:"
    )

    print(
        (
            test_neg_stage
            /
            test_neg_stage.sum()
        ).round(3)
    )


    # ========================================================
    # Data
    # ========================================================

    X_train = X_stft[
        train_indices
    ]

    y_train = y[
        train_indices
    ]


    X_val = X_stft[
        val_indices
    ]

    y_val = y[
        val_indices
    ]


    X_test = X_stft[
        test_indices
    ]

    y_test = y[
        test_indices
    ]


    print()
    print("=" * 70)
    print("DATASET SIZE")
    print("=" * 70)


    print(
        f"Train: {len(y_train)} "
        f"(Positive={np.sum(y_train == 1)}, "
        f"Negative={np.sum(y_train == 0)})"
    )


    print(
        f"Val  : {len(y_val)} "
        f"(Positive={np.sum(y_val == 1)}, "
        f"Negative={np.sum(y_val == 0)})"
    )


    print(
        f"Test : {len(y_test)} "
        f"(Positive={np.sum(y_test == 1)}, "
        f"Negative={np.sum(y_test == 0)})"
    )


    # ========================================================
    # Normalization
    # ========================================================

    train_mean = X_train.mean(
        dtype=np.float64
    )


    train_std = X_train.std(
        dtype=np.float64
    )


    print()
    print(
        "Train mean:",
        train_mean
    )

    print(
        "Train std:",
        train_std
    )


    X_train = (
        X_train - train_mean
    ) / (
        train_std + 1e-8
    )


    X_val = (
        X_val - train_mean
    ) / (
        train_std + 1e-8
    )


    X_test = (
        X_test - train_mean
    ) / (
        train_std + 1e-8
    )


    # ========================================================
    # DataLoader
    # ========================================================

    train_loader = DataLoader(
        EEGDataset(
            X_train,
            y_train
        ),
        batch_size=BATCH_SIZE,
        shuffle=True
    )


    val_loader = DataLoader(
        EEGDataset(
            X_val,
            y_val
        ),
        batch_size=BATCH_SIZE,
        shuffle=False
    )


    test_loader = DataLoader(
        EEGDataset(
            X_test,
            y_test
        ),
        batch_size=BATCH_SIZE,
        shuffle=False
    )


    # ========================================================
    # Model
    # ========================================================

    model = (
        StandardEEGResNet18(
            in_channels=6
        )
        .to(DEVICE)
    )


    # ========================================================
    # Class Weight
    # ========================================================

    class_counts = np.bincount(
        y_train,
        minlength=2
    )


    if np.any(
        class_counts == 0
    ):

        print(
            "ERROR: Train에 한쪽 class가 없습니다."
        )

        continue


    class_weights = (
        len(y_train)
        /
        (
            2.0
            *
            class_counts
        )
    )


    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE
    )


    print()
    print(
        "Class counts:",
        class_counts
    )


    print(
        "Class weights:",
        class_weights.cpu().numpy()
    )


    criterion = (
        nn.CrossEntropyLoss(
            weight=class_weights
        )
    )


    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )


    # ========================================================
    # Best model
    # ========================================================

    best_val_ba = -1.0

    best_val_auc = np.nan

    best_epoch = 0


    best_weights = (
        copy.deepcopy(
            model.state_dict()
        )
    )


    # ========================================================
    # Training
    # ========================================================

    for epoch in range(
        EPOCHS
    ):

        # ====================================================
        # TRAIN
        # ====================================================

        model.train()

        train_loss = 0.0


        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )


            optimizer.zero_grad()


            logits = model(
                xb
            )


            loss = criterion(
                logits,
                yb
            )


            loss.backward()

            optimizer.step()


            train_loss += (
                loss.item()
                *
                len(yb)
            )


        train_loss /= len(
            train_loader.dataset
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()


        val_labels = []

        val_probs = []

        val_preds = []


        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(
                    DEVICE
                )


                logits = model(
                    xb
                )


                probs = (
                    torch.softmax(
                        logits,
                        dim=1
                    )[:, 1]
                )


                preds = (
                    probs >= 0.5
                ).long()


                val_labels.extend(
                    yb.numpy()
                )


                val_probs.extend(
                    probs.cpu().numpy()
                )


                val_preds.extend(
                    preds.cpu().numpy()
                )


        val_ba = (
            balanced_accuracy_score(
                val_labels,
                val_preds
            )
        )


        try:

            val_auc = (
                roc_auc_score(
                    val_labels,
                    val_probs
                )
            )

        except ValueError:

            val_auc = np.nan


        # ====================================================
        # Best model
        # ====================================================

        if val_ba > best_val_ba:

            best_val_ba = val_ba

            best_val_auc = val_auc

            best_epoch = epoch + 1

            best_weights = (
                copy.deepcopy(
                    model.state_dict()
                )
            )


        print(
            f"Epoch {epoch + 1:3d}/{EPOCHS} | "
            f"Loss={train_loss:.4f} | "
            f"Val BA={val_ba:.4f} | "
            f"Val AUC={val_auc:.4f} | "
            f"Best={best_val_ba:.4f}"
        )


    # ========================================================
    # TEST
    # ========================================================

    model.load_state_dict(
        best_weights
    )


    model.eval()


    test_labels = []

    test_probs = []

    test_preds = []


    with torch.no_grad():

        for xb, yb in test_loader:

            xb = xb.to(
                DEVICE
            )


            logits = model(
                xb
            )


            probs = (
                torch.softmax(
                    logits,
                    dim=1
                )[:, 1]
            )


            preds = (
                probs >= 0.5
            ).long()


            test_labels.extend(
                yb.numpy()
            )


            test_probs.extend(
                probs.cpu().numpy()
            )


            test_preds.extend(
                preds.cpu().numpy()
            )


    # ========================================================
    # Test metrics
    # ========================================================

    test_ba = (
        balanced_accuracy_score(
            test_labels,
            test_preds
        )
    )


    try:

        test_auc = (
            roc_auc_score(
                test_labels,
                test_probs
            )
        )

    except ValueError:

        test_auc = np.nan


    cm = confusion_matrix(
        test_labels,
        test_preds
    )


    # ========================================================
    # Fold result
    # ========================================================

    print()
    print("=" * 70)
    print(
        f"FOLD {fold + 1} FINAL TEST"
    )
    print("=" * 70)


    print(
        "Test Negative:",
        list(
            test_neg_patients
        )
    )


    print(
        "Best Epoch:",
        best_epoch
    )


    print(
        f"Best Val BA : "
        f"{best_val_ba:.4f}"
    )


    print(
        f"Best Val AUC: "
        f"{best_val_auc:.4f}"
    )


    print(
        f"Test BA     : "
        f"{test_ba:.4f}"
    )


    print(
        f"Test AUC    : "
        f"{test_auc:.4f}"
    )


    print()
    print(
        "Confusion Matrix:"
    )

    print(
        cm
    )


    fold_results.append(
        {
            "fold":
                fold + 1,

            "best_epoch":
                best_epoch,

            "val_BA":
                best_val_ba,

            "val_AUC":
                best_val_auc,

            "test_BA":
                test_ba,

            "test_AUC":
                test_auc,

            "test_negative_patient":
                list(
                    test_neg_patients
                ),

            "train_positive_epochs":
                len(sampled_train_pos),

            "train_negative_epochs":
                len(sampled_train_neg),

            "val_positive_epochs":
                len(val_pos_indices),

            "val_negative_epochs":
                len(sampled_val_neg),

            "test_positive_epochs":
                len(test_pos_indices),

            "test_negative_epochs":
                len(sampled_test_neg),

            "CM":
                cm
        }
    )


    # ========================================================
    # Memory cleanup
    # ========================================================

    del (
        model,
        optimizer,
        criterion,
        train_loader,
        val_loader,
        test_loader,
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 28. Final Result
# ============================================================

print()
print("=" * 70)
print("FINAL RESULT - Experiment A")
print("Stage-Matched Patient-level 4-Fold")
print("=" * 70)


if len(fold_results) == 0:

    print(
        "ERROR: 유효한 fold 결과가 없습니다."
    )


else:

    val_ba = np.array(
        [
            r["val_BA"]
            for r in fold_results
        ]
    )


    val_auc = np.array(
        [
            r["val_AUC"]
            for r in fold_results
            if not np.isnan(
                r["val_AUC"]
            )
        ]
    )


    test_ba = np.array(
        [
            r["test_BA"]
            for r in fold_results
        ]
    )


    test_auc = np.array(
        [
            r["test_AUC"]
            for r in fold_results
            if not np.isnan(
                r["test_AUC"]
            )
        ]
    )


    print()


    for r in fold_results:

        print(
            f"Fold {r['fold']} | "
            f"Test Negative="
            f"{r['test_negative_patient']} | "
            f"Best Epoch={r['best_epoch']} | "
            f"Val BA={r['val_BA']:.4f} | "
            f"Val AUC={r['val_AUC']:.4f} | "
            f"Test BA={r['test_BA']:.4f} | "
            f"Test AUC={r['test_AUC']:.4f}"
        )


    print()


    print(
        f"Mean Validation BA : "
        f"{val_ba.mean():.4f} ± "
        f"{val_ba.std(ddof=1) if len(val_ba) > 1 else 0:.4f}"
    )


    if len(val_auc) > 0:

        print(
            f"Mean Validation AUC: "
            f"{val_auc.mean():.4f} ± "
            f"{val_auc.std(ddof=1) if len(val_auc) > 1 else 0:.4f}"
        )


    print(
        f"Mean Test BA       : "
        f"{test_ba.mean():.4f} ± "
        f"{test_ba.std(ddof=1) if len(test_ba) > 1 else 0:.4f}"
    )


    if len(test_auc) > 0:

        print(
            f"Mean Test AUC      : "
            f"{test_auc.mean():.4f} ± "
            f"{test_auc.std(ddof=1) if len(test_auc) > 1 else 0:.4f}"
        )


    # ========================================================
    # Save result
    # ========================================================

    result_df = pd.DataFrame(
        fold_results
    )


    result_path = os.path.join(
        BASE_FOLDER,
        "experiment_A_stage_matched_results.csv"
    )


    result_df.to_csv(
        result_path,
        index=False
    )


    print()
    print(
        "Result saved:"
    )

    print(
        result_path
    )


print()
print("=" * 70)
print("Experiment A COMPLETE")
print("=" * 70)

Experiment A
Stage-Matched Patient-Level Classification

Device: cpu

Checking folder:
/content/drive/MyDrive/Sleep_Project/example/positive_patient_data
  NPY files: 42

Checking folder:
/content/drive/MyDrive/Sleep_Project/example/positive_patient_data_3
  NPY files: 22

Checking folder:
/content/drive/MyDrive/Sleep_Project/example/negative_patient_data
  NPY files: 2

Checking folder:
/content/drive/MyDrive/Sleep_Project/example/negative_patient_data_3
  NPY files: 3

Checking folder:
/content/drive/MyDrive/Sleep_Project/example/negative_patient_data_5
  NPY files: 3

[1] Patient 확인

Positive patients: 64
Negative patients: 7

Negative patient list:
  12526_1372
  16027_15634
  2266_14353
  2833_14806
  5956_9631
  6877_17344
  9358_85

[2] Positive EEG loading

Loading 10000_17728
  EEG shape: (1, 6, 7680)
  Valid epochs: 1
  Stage distribution:
N1     0
N2     1
N3     0
REM    0
Name: count, dtype: int64

Loading 10996_10132
  EEG shape: (1, 6, 7680)
  Valid epochs: 1
  Stage dis